# Causal Wizard &mdash; Identification & Estimation

This notebook takes the config JSON and data file the [Causal Wizard](https://github.com/drawlinson/causal_wizard_app)
site generated for you and runs the actual statistical estimation &mdash; the site itself
deliberately doesn't do this (there's no server; this notebook, and your own data, are
the whole computation).

It re-derives everything from scratch &mdash; column types, the treatment/outcome
encoding, the identified causal estimand &mdash; rather than trusting anything the site
already told you, so if something doesn't check out, you'll see it here too.

**Run this from inside the repo's `notebooks/` directory** (e.g.
`cd causal_wizard_app/notebooks && jupyter notebook`) so the local `causalwizard` package
is found automatically. In Colab, the next cell installs it straight from GitHub instead.

**Output:** a `results.json` file. Open **02-results.ipynb** next, point it at the same
config/data files plus this `results.json`, and it renders the full results report.

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("causalwizard") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "causalwizard @ git+https://github.com/drawlinson/causal_wizard_app.git#subdirectory=notebooks"],
        check=True,
    )

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from causalwizard import (
    config, identification, estimation_cdpo, estimation_pdfe, counterfactuals,
    propensity, refutation, diagnostics, results_schema,
)

## 1. Inputs

Point these at the two files the site told you to download/have handy, and where you
want the `results.json` this notebook produces to be written.

In [ ]:
config_path = "study-config.json"
data_path = "data.csv"
results_path = "results.json"

## 2. Load data and re-derive the treatment/outcome columns

In [ ]:
cfg = config.load_config(config_path)
raw_df = pd.read_csv(data_path)

prepared = config.prepare_dataframe(cfg, raw_df)
train_df, test_df = config.train_test_split(prepared.df, cfg["question"]["splitTestPc"])

print(f"{len(prepared.df)} rows kept ({prepared.dropped_excluded} excluded by treatment spec, "
      f"{prepared.dropped_na} dropped for missing values); "
      f"{len(train_df)} training / {len(test_df)} held out for validation.")
print(f"Treatment: {prepared.treatment_col} (continuous={prepared.treatment_is_continuous})")
label = f", class 1 = {prepared.outcome_class1_label}" if prepared.outcome_class1_label else ""
print(f"Outcome: {prepared.outcome_col} ({prepared.outcome_effective_type}{label})")

## 3. Identification

For Causal Diagram + Potential Outcomes (CD+PO), build the DoWhy causal model from your
diagram and ask DoWhy to identify the effect &mdash; this runs the same graph algorithm the
site's own Check step used, but here independently, straight from your diagram.

For Panel Data + Fixed Effects (PD+FE) there's no DoWhy graph to identify &mdash;
"identification" is just the entity/time/covariate structure you set up on the site.

In [ ]:
method = cfg["question"]["method"]

if method == "cd+po":
    model = identification.build_causal_model(train_df, cfg["graph"], prepared.treatment_col, prepared.outcome_col)
    identified = identification.identify(model)
    estimand_names = identification.estimand_names(identified)
    print("Identified estimand(s):", estimand_names)
    if not estimand_names:
        raise ValueError(
            "DoWhy could not identify this effect from your causal diagram - there's likely an "
            "unobserved confounder blocking every backdoor path. Revisit the diagram on the site."
        )
else:
    model = identified = None
    panel = cfg["question"]["panelData"]
    print(f"Panel data: entity={panel['entity']}, time={panel['time']}, covariates={panel['covariates']}")

## 4. Estimation

Fits the model the site's Check step selected (`identification.model.key` in the config),
and computes the effect (ATE/ATT/ATC, per your chosen target units).

In [ ]:
covariate_cols = cfg["question"]["panelData"]["covariates"] if method == "pd+fe" else prepared.covariate_cols
covariate_types = {c: config.resolve_effective_type(cfg, c) for c in covariate_cols}
outcome_is_binary = prepared.outcome_effective_type == "categorical"

if method == "cd+po":
    method_key = cfg["identification"]["model"]["key"]
    est = estimation_cdpo.estimate(
        model, identified, method_key, train_df, prepared.treatment_col, prepared.outcome_col,
        covariate_cols, covariate_types, outcome_is_binary, cfg["question"]["effect"],
    )
    estimand_info = identification.estimand_info(identified, est.estimand_type)
    backdoor_vars = identification.estimand_variables(identified, est.estimand_type)
else:
    panel = cfg["question"]["panelData"]
    est = estimation_pdfe.estimate(
        train_df, panel["entity"], panel["time"], prepared.treatment_col, prepared.outcome_col,
        covariate_cols, covariate_types,
    )
    method_key = "panel.fixed_effects"
    estimand_info = {"expression": None, "assumptions": None}
    backdoor_vars = []

print(f"Effect ({cfg['question']['effect'].upper()}): {est.effect:.4g}")

## 5. Validation / refutation

In [ ]:
if method == "cd+po":
    if est.dowhy_estimate is not None:
        validation = refutation.run_cdpo_refutation(model, identified, est.dowhy_estimate)
        accept = refutation.cdpo_accept(validation)
    else:
        validation, accept = None, None
        print("This estimator (own linear regression/GLM) has no DoWhy refuters to run - "
              "same scope as the site's own validation section.")
else:
    validation = refutation.run_pdfe_validation(est.z_pvalue, est.f_pvalue)
    accept = refutation.pdfe_accept(validation)

print("Validation:", validation)
print("Accept:", accept)

## 6. Counterfactuals and held-out generalization

Only estimators with a "do-operator" predict function support these (CD+PO's own linear
regression/GLM, and PD+FE always) &mdash; matches the site's own "regression models only" note.

In [ ]:
treatment_spec = cfg["question"]["treatmentSpec"]
cf_control = treatment_spec.get("counterfactualLower")
cf_treated = treatment_spec.get("counterfactualUpper")
cf_control = 0.0 if cf_control is None else cf_control
cf_treated = 1.0 if cf_treated is None else cf_treated

cf_table = counterfactuals.counterfactual_table(
    train_df, prepared.treatment_col, est.predict, prepared.treatment_is_continuous, cf_control, cf_treated,
)
generalization = counterfactuals.generalization_predictions(
    test_df, est.predict, prepared.treatment_col, prepared.outcome_col
)
train_preds = counterfactuals.train_predictions(train_df, est.predict, cf_control, cf_treated)

print("Counterfactual table:", cf_table)
print("Generalization sample count:", None if generalization is None else len(generalization["actual"]))

## 7. Propensity diagnostics\n\nOnly meaningful for the propensity-based CD+PO estimators.

In [ ]:
propensity_analysis = None
propensity_estimators = ("propensity_score_weighting", "propensity_score_matching", "propensity_score_stratification")
if method == "cd+po" and est.estimator in propensity_estimators:
    ps = propensity.extract_propensity_scores(est.dowhy_estimate)
    treatment_binary = train_df[prepared.treatment_col].to_numpy()
    distribution = propensity.positivity_distribution(ps, treatment_binary)
    balance = propensity.covariate_balance(train_df, backdoor_vars, covariate_types, treatment_binary, ps)
    propensity_analysis = {"distribution": distribution, "covariate_balance": balance}
    print("Positivity - rows in the unreliable tail bins:", distribution["distribution_bad"])
    print("Covariate balance:", balance)
else:
    print("Not a propensity-based estimator - no positivity/balance diagnostics.")

## 8. Sample counts, contingency table, modelling statements

In [ ]:
contingency = diagnostics.contingency_table(
    train_df, prepared.treatment_col, prepared.outcome_col, outcome_is_binary, prepared.outcome_class1_label,
)
sample_counts = contingency["total"]

modelling_statements = diagnostics.modelling_statements(
    method, prepared.treatment_col, prepared.outcome_col, prepared.treatment_is_continuous,
    prepared.outcome_effective_type, prepared.outcome_class1_label, sample_counts,
    est.estimand_type if method == "cd+po" else None,
    estimand_info["expression"],
    est.estimator if method == "cd+po" else "fixed_effects",
    cfg["question"]["panelData"] if method == "pd+fe" else None,
)
print(sample_counts)
print(modelling_statements)

## 9. Save `results.json`\n\nEverything notebook 2 needs, as plain JSON - no fitted model objects.

In [ ]:
results = results_schema.build_results(
    method=method, treatment_col=prepared.treatment_col, outcome_col=prepared.outcome_col,
    treatment_is_continuous=prepared.treatment_is_continuous, outcome_effective_type=prepared.outcome_effective_type,
    outcome_class1_label=prepared.outcome_class1_label, target_units=cfg["question"]["effect"], effect=est.effect,
    estimand_type=est.estimand_type if method == "cd+po" else None,
    estimand_expression=estimand_info["expression"],
    estimator_name=est.estimator if method == "cd+po" else "fixed_effects",
    method_key=method_key, assumptions=estimand_info["assumptions"], validation=validation, accept=accept,
    counterfactuals=cf_table, counterfactual_control_value=cf_control, counterfactual_treated_value=cf_treated,
    generalization=generalization, contingency=contingency, sample_counts=sample_counts,
    propensity_analysis=propensity_analysis, modelling_statements=modelling_statements,
    panel_data=cfg["question"]["panelData"] if method == "pd+fe" else None,
    dropped_excluded=prepared.dropped_excluded, dropped_na=prepared.dropped_na,
    train_predictions=train_preds, estimand_variables=backdoor_vars,
)
results_schema.save_results(results, results_path)
print(f"Saved {results_path}")

## Next step

Open **02-results.ipynb**, point it at the same `config_path`/`data_path` plus this
`results_path`, and run it to see the full results report &mdash; findings, plots,
refutation, assumptions and more.